# 4. Modeling & Evaluation

This notebook trains and evaluates wildfire ignition prediction models through iterative feature addition:

| Model | Features | Metric |
|-------|----------|--------|
| Logistic Regression (baseline) | Weather only | ROC-AUC, PR-AUC |
| XGBoost | Weather only | ROC-AUC, PR-AUC |
| Logistic Regression + Elevation | Weather + SRTM | ROC-AUC, PR-AUC |
| Logistic Regression + NDVI | Weather + SRTM + MODIS NDVI | ROC-AUC, PR-AUC |

> **Prerequisite:** Run Notebooks 01-03 first.

## Train/Test Split (2019-2022 / 2023)

In [ ]:
# Train: 2019-01-01 to 2022-12-31
train_df = model_df[model_df["date"] < "2023-01-01"].copy()

# Test: 2023
test_df = model_df[model_df["date"] >= "2023-01-01"].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train positives:", train_df["ignition"].sum())
print("Test positives:", test_df["ignition"].sum())

## Baseline: Logistic Regression (Weather Features)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

features = [
    "temp_7d_mean",
    "wind_7d_mean",
    "vpd_7d_mean",
    "precip_14d_sum"
]

X_train = train_df[features]
y_train = train_df["ignition"]

X_test = test_df[features]
y_test = test_df["ignition"]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Class weight balanced (important)
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

# Predictions
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

roc = roc_auc_score(y_test, y_pred_proba)
pr = average_precision_score(y_test, y_pred_proba)

print("ROC-AUC:", roc)
print("PR-AUC:", pr)

In [ ]:
test_df = test_df.copy()
test_df["pred_prob"] = y_pred_proba

In [ ]:
test_df = test_df.sort_values("pred_prob", ascending=False)

## Top-K Capture Rate Analysis

In [ ]:
top_5_percent = int(0.05 * len(test_df))

top_slice = test_df.iloc[:top_5_percent]

captured_ignitions = top_slice["ignition"].sum()
total_ignitions = test_df["ignition"].sum()

print("Total ignitions in 2023:", total_ignitions)
print("Ignitions in top 5% risk:", captured_ignitions)
print("Capture rate:", captured_ignitions / total_ignitions)

## XGBoost Classifier

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

roc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
pr_xgb = average_precision_score(y_test, y_pred_proba_xgb)

print("XGB ROC-AUC:", roc_xgb)
print("XGB PR-AUC:", pr_xgb)

In [ ]:
test_df["pred_prob_xgb"] = y_pred_proba_xgb
test_df = test_df.sort_values("pred_prob_xgb", ascending=False)

top_slice_xgb = test_df.iloc[:top_5_percent]

captured_xgb = top_slice_xgb["ignition"].sum()

print("Top 5% capture (XGB):", captured_xgb / total_ignitions)

## Adding Elevation Data (SRTM)

In [ ]:
# Load SRTM elevation
elevation_img = ee.Image("USGS/SRTMGL1_003")

In [ ]:
elevation_sample = elevation_img.sampleRegions(
    collection=grid_points,
    scale=90,
    geometries=False
)

print("Elevation samples:", elevation_sample.size().getInfo())

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=elevation_sample,
    description="elevation_grid_cells",
    folder="Wildfire_Project",
    fileNamePrefix="elevation_grid_cells",
    fileFormat="CSV"
)

task.start()
print("Elevation export started.")

In [ ]:
elev_df = pd.read_csv("/content/drive/MyDrive/Wildfire_Project/elevation_grid_cells.csv")

print("Rows:", len(elev_df))
print(elev_df.head())
print(elev_df.columns)

In [ ]:
elev_df = elev_df.drop(columns=["system:index", ".geo"])
print(elev_df.head())

In [ ]:
model_df = model_df.merge(
    elev_df,
    on=["lat_bin", "lon_bin"],
    how="left"
)

print("Rows after elevation merge:", len(model_df))
print("Missing elevation:", model_df["elevation"].isna().sum())

## Logistic Regression + Elevation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

features_with_elev = [
    "temp_7d_mean",
    "wind_7d_mean",
    "vpd_7d_mean",
    "precip_14d_sum",
    "elevation"
]

X_train = train_df.merge(
    elev_df,
    on=["lat_bin", "lon_bin"],
    how="left"
)[features_with_elev]

y_train = train_df["ignition"]

X_test = test_df.merge(
    elev_df,
    on=["lat_bin", "lon_bin"],
    how="left"
)[features_with_elev]

y_test = test_df["ignition"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_elev = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model_elev.fit(X_train_scaled, y_train)

y_pred_proba_elev = model_elev.predict_proba(X_test_scaled)[:, 1]

roc_elev = roc_auc_score(y_test, y_pred_proba_elev)
pr_elev = average_precision_score(y_test, y_pred_proba_elev)

print("ROC-AUC (with elevation):", roc_elev)
print("PR-AUC (with elevation):", pr_elev)

In [ ]:
test_df["pred_prob_elev"] = y_pred_proba_elev
test_df = test_df.sort_values("pred_prob_elev", ascending=False)

top_slice_elev = test_df.iloc[:top_5_percent]

captured_elev = top_slice_elev["ignition"].sum()

print("Top 5% capture (with elevation):", captured_elev / total_ignitions)

## Adding NDVI Data (MODIS)

In [ ]:
# Load MODIS NDVI collection
ndvi_collection = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .select("NDVI")
    .filterDate("2019-01-01", "2023-12-31")
    .filterBounds(california)
)

print("NDVI images:", ndvi_collection.size().getInfo())

In [ ]:
# Function to compute monthly NDVI mean
def monthly_ndvi(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    monthly_img = (
        ndvi_collection
        .filterDate(start, end)
        .mean()
        .set("year", year)
        .set("month", month)
        .set("date", start.format("YYYY-MM-dd"))
    )

    return monthly_img

# Create list of years and months
years = list(range(2019, 2024))
months = list(range(1, 13))

monthly_images = []

for y in years:
    for m in months:
        monthly_images.append(monthly_ndvi(y, m))

monthly_ndvi_collection = ee.ImageCollection(monthly_images)

print("Monthly NDVI images:", monthly_ndvi_collection.size().getInfo())

In [ ]:
# Function to sample one monthly image
def sample_ndvi(image):
    date_str = image.get("date")

    sampled = image.sampleRegions(
        collection=grid_points,
        scale=1000,
        geometries=False
    )

    return sampled.map(lambda f: f.set("date", date_str))

ndvi_fc = monthly_ndvi_collection.map(sample_ndvi).flatten()

print("Total NDVI samples:", ndvi_fc.size().getInfo())

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=ndvi_fc,
    description="monthly_ndvi_grid",
    folder="Wildfire_Project",
    fileNamePrefix="monthly_ndvi_grid",
    fileFormat="CSV"
)

task.start()
print("NDVI export started.")

In [ ]:
ndvi_df = pd.read_csv("/content/drive/MyDrive/Wildfire_Project/monthly_ndvi_grid.csv")

print("Rows:", len(ndvi_df))
print(ndvi_df.head())
print(ndvi_df.columns)

In [ ]:
ndvi_df = ndvi_df.drop(columns=["system:index", ".geo"])

# Convert date
ndvi_df["date"] = pd.to_datetime(ndvi_df["date"])

# Extract year and month
ndvi_df["year"] = ndvi_df["date"].dt.year
ndvi_df["month"] = ndvi_df["date"].dt.month

# Scale NDVI
ndvi_df["ndvi"] = ndvi_df["NDVI"] / 10000

# Keep only needed columns
ndvi_df = ndvi_df[["lat_bin", "lon_bin", "year", "month", "ndvi"]]

print(ndvi_df.head())
print("Rows:", len(ndvi_df))

In [ ]:
model_df["year"] = model_df["date"].dt.year
model_df["month"] = model_df["date"].dt.month

In [ ]:
model_df = model_df.merge(
    ndvi_df,
    on=["lat_bin", "lon_bin", "year", "month"],
    how="left"
)

print("Rows after NDVI merge:", len(model_df))
print("Missing NDVI:", model_df["ndvi"].isna().sum())

In [ ]:
model_df["ndvi"] = model_df["ndvi"].fillna(0)

## Final Model: Logistic Regression + All Features

In [ ]:
features_with_ndvi = [
    "temp_7d_mean",
    "wind_7d_mean",
    "vpd_7d_mean",
    "precip_14d_sum",
    "elevation",
    "ndvi"
]

# Recreate train/test with NDVI included
train_df = model_df[model_df["date"] < "2023-01-01"].copy()
test_df = model_df[model_df["date"] >= "2023-01-01"].copy()

X_train = train_df[features_with_ndvi]
y_train = train_df["ignition"]

X_test = test_df[features_with_ndvi]
y_test = test_df["ignition"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_ndvi = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model_ndvi.fit(X_train_scaled, y_train)

y_pred_proba_ndvi = model_ndvi.predict_proba(X_test_scaled)[:, 1]

roc_ndvi = roc_auc_score(y_test, y_pred_proba_ndvi)
pr_ndvi = average_precision_score(y_test, y_pred_proba_ndvi)

print("ROC-AUC (with NDVI):", roc_ndvi)
print("PR-AUC (with NDVI):", pr_ndvi)

In [ ]:
test_df["pred_prob_ndvi"] = y_pred_proba_ndvi
test_df = test_df.sort_values("pred_prob_ndvi", ascending=False)

top_slice_ndvi = test_df.iloc[:top_5_percent]

captured_ndvi = top_slice_ndvi["ignition"].sum()

print("Top 5% capture (with NDVI):", captured_ndvi / total_ignitions)

## Capture Rate Comparison

In [ ]:
for pct in [0.05, 0.10, 0.20]:
    cutoff = int(pct * len(test_df))
    slice_df = test_df.iloc[:cutoff]
    capture = slice_df["ignition"].sum() / total_ignitions
    print(f"Top {int(pct*100)}% capture:", capture)